# Vibe Quest S10 · Polisemanticidad, autoencoders dispersos y "dirigir" características

**Pregunta que investigo (pregunta 3):** la polisemanticidad y los esfuerzos por lograr características monosemánticas con autoencoders en modelos grandes. ¿Cómo puede la capacidad de *dirigir* modelos de lenguaje, manipulando esas características internas, transformar la auditoría, la alineación y la detección de sesgos, y qué riesgos éticos plantea esa intervención detallada?

Jorge Chamorro · INF659 Técnicas Avanzadas de Data Mining y Sistemas Inteligentes · PUCP 2026-2

## Cómo trabajé
Yo definí el planteamiento: no puedo entrenar un modelo de lenguaje grande en una clase, así que reproduzco en pequeño el mecanismo que explican las fuentes (superposición, autoencoder disperso, steering) en un mundo sintético donde **conozco los rasgos verdaderos** y puedo comprobar si el método los recupera. Usé un asistente de IA como apoyo para buscar y leer fuentes, escribir el código y probarlo en Colab. Las decisiones de diseño, las métricas y la lectura de los resultados las revisé yo, y cada resultado numérico de este cuaderno sale de ejecutarlo, no de las fuentes.

## Qué pruebo (tres hipótesis que puedo falsar)
1. **Superposición:** si los rasgos son *esparsos* (rara vez aparecen a la vez), un modelo con menos neuronas que rasgos los comprime en direcciones superpuestas y las neuronas quedan mezcladas (polisemánticas). Si los rasgos son densos, no puede.
2. **Recuperación:** un autoencoder disperso (SAE) con más latentes que neuronas y penalización L1 recupera las direcciones verdaderas; sin L1 o con L1 excesiva no.
3. **Dirección (steering):** fijar un latente del SAE cambia casi solo el rasgo objetivo; fijar una neurona cambia varios rasgos a la vez.

Lo que **no** puedo probar aquí: que esto escale a Claude o a otro modelo real, ni que las características de seguridad reales sean confiables. Eso lo discuto al final con las fuentes y sin inventar cifras.

## Cómo leer este cuaderno
Cada sección se titula `N · Nombre`. Antes de cada celda de código hay una celda de texto que explica qué hace y qué debería verse. Cada celda que muestra una tabla, un gráfico o guarda un archivo está sola, para poder ejecutarla y mirarla por separado. Ejecuta las celdas en orden (tarda unos 3 a 4 minutos en CPU).

## 1 · Librerías
Aquí solo se importan las librerías: `torch` para las redes (modelo de juguete y SAE), `numpy` y `pandas` para los números y las tablas, `matplotlib` para los gráficos, `ipywidgets` para los controles interactivos y `Path` para guardar archivos. No imprime nada.

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

## 2 · Parámetros y carpetas
### 2 · a · Parámetros del experimento
El mundo tiene 12 rasgos y el modelo de juguete solo 4 neuronas: es imposible dar una neurona a cada rasgo, así que si el modelo quiere representarlos todos tiene que superponerlos. El SAE tiene 24 latentes (factor 6, un "diccionario sobrecompleto" como en Cunningham et al., 2023). La *esparsidad* es la probabilidad de que un rasgo esté apagado: 0.5 es denso, 0.97 es muy esparso. `α` es el peso de la penalización L1 del SAE; sus valores los fijé tras una exploración previa (ver sección 16 · limitaciones). Fijo también la semilla para que los resultados se repitan. No imprime nada.

In [ ]:
# ---------------- Parámetros del experimento ----------------
SEMILLA = 0
N_RASGOS = 12            # rasgos "verdaderos" que genera el mundo sintético
N_DIM = 4                # neuronas de la capa oculta del modelo de juguete (menos que rasgos)
K_DICT = 24              # latentes del autoencoder disperso (factor de expansión 6)
ESPARSIDADES = [0.5, 0.9, 0.97]   # probabilidad de que un rasgo esté apagado
ALPHAS = [0.0, 0.1, 0.3, 1.0]     # peso de la penalización L1 del autoencoder disperso
PASOS_MODELO = 4000
PASOS_SAE = 5000
LOTE = 2048

torch.manual_seed(SEMILLA)
np.random.seed(SEMILLA)

### 2 · b · Carpetas de salida
Crea `resultados/datos` y `resultados/capturas`, donde las secciones 9, 10 y 13 guardan los CSV y las figuras. En Colab aparecen en el panel de archivos de la izquierda.

In [ ]:
# carpetas donde el cuaderno guarda sus resultados
DATOS = Path("resultados/datos")
CAPTURAS = Path("resultados/capturas")
DATOS.mkdir(parents=True, exist_ok=True)
CAPTURAS.mkdir(parents=True, exist_ok=True)

## 3 · Datos sintéticos
### 3 · a · Generador de datos
`muestrear` crea vectores de 12 rasgos. Cada rasgo está encendido con probabilidad `1 − esparsidad` y, si está encendido, vale un número entre 0 y 1. No hay datos externos: todo se genera con semillas fijas.

In [ ]:
def muestrear(n, esparsidad, generador):
    """Cada rasgo está activo con prob. (1 - esparsidad); si está activo, vale U(0,1)."""
    activo = (torch.rand(n, N_RASGOS, generator=generador) > esparsidad).float()
    valor = torch.rand(n, N_RASGOS, generator=generador)
    return activo * valor

### 3 · b · ¿Cuántos rasgos están activos a la vez?
Imprime, para cada esparsidad, el promedio de rasgos activos por muestra y el porcentaje de muestras con dos o más rasgos activos a la vez. Es la base de la superposición: si casi nunca coinciden dos rasgos, el modelo puede compartir una dirección entre ellos con poca interferencia.

In [ ]:
g = torch.Generator().manual_seed(SEMILLA)
filas_datos = []
for S in ESPARSIDADES:
    x = muestrear(20000, S, g)
    activos = (x > 0).sum(1).float()
    filas_datos.append({"esparsidad": S, "rasgos_activos_promedio": activos.mean().item(),
                        "% muestras con 2+ activos": 100 * (activos >= 2).float().mean().item()})
print(pd.DataFrame(filas_datos).round(2).to_string(index=False))

### 3 · c · Gráfico: distribución de rasgos activos por muestra
Histograma del número de rasgos activos por muestra para las tres esparsidades. Con 0.97 la mayoría de las muestras tiene 0 o 1 rasgo activo; con 0.5 suelen ser unos 6.

In [ ]:
g = torch.Generator().manual_seed(SEMILLA)
fig_datos, ejes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for eje, S in zip(ejes, ESPARSIDADES):
    activos = (muestrear(20000, S, g) > 0).sum(1)
    eje.hist(activos.numpy(), bins=range(0, N_RASGOS + 2), color="#2a9d8f", align="left")
    eje.set_title(f"esparsidad {S}"); eje.set_xlabel("rasgos activos por muestra")
ejes[0].set_ylabel("muestras")
plt.tight_layout(); plt.show()

## 4 · Modelo de juguete
`ModeloJuguete` sigue el planteamiento de Elhage et al. (2022): comprime los 12 rasgos a 4 números con una matriz `W` (4×12) y los reconstruye con `Wᵀ` más una ReLU. La capa oculta `h` (4 valores) son "las neuronas" que quiero interpretar. Es la capa que luego le entrego al SAE. La celda solo define la clase.

In [ ]:
class ModeloJuguete(nn.Module):
    """x (12 rasgos) -> h = W x (4 neuronas) -> ReLU(W^T h + b): comprime y reconstruye."""

    def __init__(self):
        super().__init__()
        self.W = nn.Parameter(torch.randn(N_DIM, N_RASGOS) * 0.5)
        self.b = nn.Parameter(torch.zeros(N_RASGOS))

    def oculta(self, x):
        return x @ self.W.T

    def salida(self, h):
        return torch.relu(h @ self.W + self.b)

    def forward(self, x):
        return self.salida(self.oculta(x))

## 5 · Autoencoder disperso (SAE)
`SAE` toma la activación `h` (4 valores) y la expande a 24 latentes `f` con una ReLU; luego la reconstruye con un decodificador lineal. Normalizo las columnas del decodificador (norma 1) para que la penalización L1 no se pueda evadir achicando `f` y agrandando las columnas. Cada columna del decodificador es una "dirección" candidata a ser un rasgo verdadero. La celda solo define la clase.

In [ ]:
class SAE(nn.Module):
    """Autoencoder disperso: h (4) -> ReLU -> f (24 latentes) -> h' (4)."""

    def __init__(self):
        super().__init__()
        self.enc = nn.Linear(N_DIM, K_DICT)
        self.dec = nn.Parameter(torch.randn(N_DIM, K_DICT))
        self.bd = nn.Parameter(torch.zeros(N_DIM))

    def direcciones(self):
        # columnas del decodificador con norma 1: así L1 no se "evade" achicando f
        return self.dec / self.dec.norm(dim=0, keepdim=True)

    def codificar(self, h):
        return torch.relu(self.enc(h - self.bd))

    def decodificar(self, f):
        return f @ self.direcciones().T + self.bd

    def forward(self, h):
        f = self.codificar(h)
        return self.decodificar(f), f

## 6 · Entrenamiento
- **Modelo:** minimiza el error cuadrático entre el rasgo original y su reconstrucción.
- **SAE:** pérdida = error de reconstrucción (suma de cuadrados) + `α · Σ activaciones`. Es la misma forma que usan Cunningham et al. (2023) y Anthropic (2023, 2024), a escala diminuta. La penalización empuja a que pocos latentes se activen a la vez (dispersión).

Ambas funciones usan Adam con una tasa de aprendizaje que decae. La celda solo define las funciones; el entrenamiento real ocurre en la sección 8.

In [ ]:
def entrenar_modelo(esparsidad, semilla=SEMILLA):
    torch.manual_seed(semilla)
    g = torch.Generator().manual_seed(semilla)
    m = ModeloJuguete()
    opt = torch.optim.Adam(m.parameters(), lr=0.005)
    sch = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=0.999)
    for _ in range(PASOS_MODELO):
        x = muestrear(LOTE, esparsidad, g)
        perdida = ((m(x) - x) ** 2).mean()
        opt.zero_grad(); perdida.backward(); opt.step(); sch.step()
    return m


def entrenar_sae(modelo, esparsidad, alpha, semilla=SEMILLA):
    torch.manual_seed(semilla)
    g = torch.Generator().manual_seed(semilla + 1)
    sae = SAE()
    opt = torch.optim.Adam(sae.parameters(), lr=0.002)
    sch = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=0.9995)
    for _ in range(PASOS_SAE):
        with torch.no_grad():
            h = modelo.oculta(muestrear(LOTE, esparsidad, g))
        h_rec, f = sae(h)
        rec = ((h_rec - h) ** 2).sum(1).mean()
        disp = f.sum(1).mean()
        perdida = rec + alpha * disp
        opt.zero_grad(); perdida.backward(); opt.step(); sch.step()
    return sae

## 7 · Métricas y dirección
### 7 · a · Evaluar
- **representados:** cuántos de los 12 rasgos tienen una columna de `W` de norma mayor que 0.5 (el modelo realmente los guarda).
- **recupera:** fracción de esos rasgos para los que algún latente del SAE tiene coseno mayor que 0.9 con la dirección verdadera.
- **pureza:** correlación máxima de cada unidad (neurona o latente) con algún rasgo verdadero, promediada. Pureza 1 significa que la unidad vale exactamente un rasgo (monosemántica).
- **l0:** latentes activos por muestra. **vivos:** latentes que se activan alguna vez. **error_rel:** error de reconstrucción relativo.

In [ ]:
def corr_max(A, B):
    """Para cada columna de A, mayor correlación con alguna columna de B (0 si es constante)."""
    A = (A - A.mean(0)) / (A.std(0) + 1e-8)
    B = (B - B.mean(0)) / (B.std(0) + 1e-8)
    C = (A.T @ B) / A.shape[0]
    return C.max(1).values, C


@torch.no_grad()
def evaluar(modelo, sae, esparsidad, n=5000, semilla=123):
    g = torch.Generator().manual_seed(semilla)
    x = muestrear(n, esparsidad, g)
    h = modelo.oculta(x)
    f = sae.codificar(h)
    h_rec = sae.decodificar(f)
    W = modelo.W.detach()
    normas = W.norm(dim=0)
    repr_ = normas > 0.5                                   # rasgos que el modelo sí representa
    Wn = W / (normas + 1e-8)
    cos = Wn.T @ sae.direcciones()                         # (12 rasgos x 24 latentes)
    recupera = (cos.max(1).values > 0.9)[repr_].float().mean().item() if repr_.any() else float("nan")
    # pureza: correlación máxima de cada unidad (neurona / latente) con algún rasgo verdadero
    p_neu, _ = corr_max(h, x)
    vivos = (f > 1e-6).float().mean(0) > 0.005             # latentes que se activan alguna vez
    p_lat, _ = corr_max(f[:, vivos], x) if vivos.any() else (torch.zeros(1), None)
    err = ((h_rec - h) ** 2).sum(1).mean().item() / (h ** 2).sum(1).mean().item()
    return dict(
        representados=int(repr_.sum()),
        recupera=recupera,
        pureza_neurona=p_neu.mean().item(),
        pureza_latente=p_lat.mean().item(),
        l0=(f > 1e-6).float().sum(1).mean().item(),
        vivos=int(vivos.sum()),
        error_rel=err,
    )

### 7 · b · Dirigir (steering)
Fijo una unidad a `factor × su activación máxima` (como el *clamp* de Anthropic), recalculo la salida y mido el cambio medio en los 12 rasgos. Puedo hacerlo por el **latente del SAE** más alineado al rasgo o por la **neurona** que más pesa en él. **Selectividad** = cambio en el rasgo objetivo ÷ suma de los cambios en los demás; más alta es más limpia, y mayor que 1 significa que el objetivo se movió más que todos los otros juntos.

In [ ]:
@torch.no_grad()
def dirigir(modelo, sae, esparsidad, rasgo, factor, metodo, n=5000, semilla=321):
    """Fija (clamp) una unidad a factor x su activación máxima y mide el cambio medio en las 12 salidas."""
    g = torch.Generator().manual_seed(semilla)
    x = muestrear(n, esparsidad, g)
    h = modelo.oculta(x)
    base = modelo.salida(h)
    if metodo == "SAE":
        f = sae.codificar(h)
        W = modelo.W.detach()
        Wn = W / (W.norm(dim=0) + 1e-8)
        j = int((Wn[:, rasgo] @ sae.direcciones()).argmax())      # latente más alineado al rasgo
        f2 = f.clone()
        f2[:, j] = factor * f[:, j].max()
        h2 = sae.decodificar(f2)
    else:
        k = int(modelo.W.detach()[:, rasgo].abs().argmax())       # neurona que más pesa en ese rasgo
        h2 = h.clone()
        h2[:, k] = factor * h[:, k].abs().max()
    delta = (modelo.salida(h2) - base).mean(0)
    otros = torch.ones(N_RASGOS, dtype=torch.bool); otros[rasgo] = False
    selectividad = delta[rasgo].abs().item() / (delta[otros].abs().sum().item() + 1e-8)
    return delta.numpy(), selectividad

## 8 · Entrenamiento previo
Entreno todo una vez: 3 modelos (esparsidad 0.5, 0.9 y 0.97) y 12 SAE (4 valores de α por modelo). Los guardo en diccionarios para que los controles interactivos solo *consulten* y respondan al instante. Imprime cuánto tardó (unos 2 a 3 minutos en CPU).

In [ ]:
t0 = time.time()
modelos = {S: entrenar_modelo(S) for S in ESPARSIDADES}
saes = {(S, a): entrenar_sae(modelos[S], S, a) for S in ESPARSIDADES for a in ALPHAS}
print(f"Entrenados {len(modelos)} modelos y {len(saes)} SAE en {time.time() - t0:.0f} s")

## 9 · Resultados principales
### 9 · a · Tabla completa
Cada fila es un par (esparsidad del mundo, α del SAE). Con α = 0 pruebo qué pasa sin penalización; con α = 1, qué pasa si me paso. Con esto ya se pueden comprobar las hipótesis 1 y 2 con números.

In [ ]:
filas = []
for S in ESPARSIDADES:
    for a in ALPHAS:
        filas.append({"esparsidad": S, "alpha": a, **evaluar(modelos[S], saes[(S, a)], S)})
tabla = pd.DataFrame(filas)
display(tabla.round(3))

### 9 · b · Función auxiliar
`fila(S, a)` devuelve la fila de la tabla para una esparsidad y un α. Solo se define aquí y se usa en la celda siguiente.

In [ ]:
def fila(S, a):
    return tabla[(tabla.esparsidad == S) & (tabla.alpha == a)].iloc[0]

### 9 · c · Comprobaciones automáticas
Tres `assert` que protegen el texto del cuaderno: con α = 0.3 y mundo esparso el SAE debe recuperar al menos 90 % de los rasgos y sus latentes deben ser más puros que las neuronas; con rasgos densos no debe recuperar casi nada. Si alguno falla, el resultado cambió y hay que revisar los textos.

In [ ]:
for S in (0.9, 0.97):
    f = fila(S, 0.3)
    assert f.recupera >= 0.9, f"SAE (alpha 0.3, S={S}) debía recuperar >=90 % de rasgos: {f.recupera:.2f}"
    assert f.pureza_latente > f.pureza_neurona + 0.2, "los latentes debían ser más puros que las neuronas"
assert fila(0.5, 0.3).recupera < 0.5, "con rasgos densos no debería recuperar casi nada"
print("Comprobaciones superadas")

### 9 · d · Gráfico: efecto de α
Izquierda: fracción de rasgos recuperados según α. Derecha: pureza de los latentes (línea) frente a la de las neuronas (punteado). Se ve el máximo en α = 0.3 para los mundos esparsos y que en el mundo denso (0.5) no hay mejora.

In [ ]:
fig_alpha, ejes = plt.subplots(1, 2, figsize=(11, 3.8))
for S, g_ in tabla.groupby("esparsidad"):
    ejes[0].plot(g_.alpha.astype(str), g_.recupera, marker="o", label=f"esparsidad {S}")
    ejes[1].plot(g_.alpha.astype(str), g_.pureza_latente, marker="o", label=f"latentes, esp. {S}")
    ejes[1].axhline(g_.pureza_neurona.iloc[0], ls=":", color=ejes[1].lines[-1].get_color())
ejes[0].set_title("Rasgos recuperados por el SAE"); ejes[0].set_xlabel("α (L1)"); ejes[0].set_ylabel("fracción recuperada"); ejes[0].legend()
ejes[1].set_title("Pureza: latentes (línea) y neuronas (punteado)"); ejes[1].set_xlabel("α (L1)"); ejes[1].set_ylabel("pureza"); ejes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

### 9 · e · Guardar la tabla y el gráfico
Guarda `resultados_tabla_sae.csv` y `fig1_alpha_recuperacion_pureza.png` en `resultados/`. En Colab quedan en la carpeta del entorno (panel de archivos de la izquierda); descárgalos si los necesitas.

In [ ]:
tabla.round(4).to_csv(DATOS / "resultados_tabla_sae.csv", index=False)
fig_alpha.savefig(CAPTURAS / "fig1_alpha_recuperacion_pureza.png", dpi=150)
print("Guardados:", DATOS / "resultados_tabla_sae.csv", "y", CAPTURAS / "fig1_alpha_recuperacion_pureza.png")

## 10 · Robustez: ¿fue suerte de una semilla?
### 10 · a · Función que repite el experimento
Repite el experimento principal (esparsidad 0.97, α = 0.3) con otras semillas: cambia la inicialización del modelo y del SAE. Además de las métricas, calcula la selectividad promedio del steering (factor 3) para el SAE y para la neurona, solo en los rasgos que el modelo representa. La celda solo define la función.

In [ ]:
def repetir(S, a, semillas):
    out = []
    for s in semillas:
        m = entrenar_modelo(S, semilla=s)
        z = entrenar_sae(m, S, a, semilla=s)
        r = evaluar(m, z, S)
        sels_sae, sels_neu = [], []
        for i in range(N_RASGOS):
            if m.W.detach()[:, i].norm() > 0.5:
                sels_sae.append(dirigir(m, z, S, i, 3.0, "SAE")[1])
                sels_neu.append(dirigir(m, z, S, i, 3.0, "Neurona")[1])
        out.append({"semilla": s, **r, "selectividad_SAE": np.mean(sels_sae), "selectividad_neurona": np.mean(sels_neu)})
    return pd.DataFrame(out)

### 10 · b · Resultado por semilla
Entrena 5 modelos y 5 SAE nuevos (tarda alrededor de 1 minuto) y muestra una fila por semilla. Si `recupera` varía mucho entre filas, el resultado de la semilla 0 no es representativo.

In [ ]:
robustez = repetir(0.97, 0.3, [1, 2, 3, 4, 5])
display(robustez.round(3))

### 10 · c · Promedio y desviación
Resume las 5 semillas. **Esta es la cifra que reporto:** la recuperación perfecta de la semilla 0 fue el mejor caso, el promedio es menor.

In [ ]:
resumen = robustez[["recupera", "pureza_neurona", "pureza_latente", "selectividad_SAE", "selectividad_neurona"]].agg(["mean", "std"])
print(resumen.round(3))

### 10 · d · Guardar la robustez
Guarda `resultados_robustez_semillas.csv` y `resultados_robustez_resumen.csv` en `resultados/datos/`.

In [ ]:
robustez.round(4).to_csv(DATOS / "resultados_robustez_semillas.csv", index=False)
resumen.round(4).to_csv(DATOS / "resultados_robustez_resumen.csv")
print("Guardados en", DATOS)

## 11 · Panel interactivo A · ¿Qué ve el SAE que las neuronas no muestran?
Elige la esparsidad del mundo y el α del SAE.
- **Izquierda:** pesos `|W|` (neuronas × rasgos). Si cada neurona tiene varios cuadros brillantes, es polisemántica.
- **Centro:** coseno entre cada rasgo verdadero y los latentes del SAE. Un cuadro brillante por fila significa que el SAE separó ese rasgo.
- **Derecha:** pureza media de neuronas frente a latentes.

La línea de texto de abajo da recupera, latentes vivos, L0 y error. Prueba esparsidad 0.97 con α = 0 y con α = 0.3, y luego 0.5.

### 11 · a · Función que dibuja el panel
Define `panel_a(S, a)`: toma el modelo y el SAE ya entrenados para esa esparsidad y ese α, y dibuja los tres gráficos. Esta celda solo define la función.

In [ ]:
def panel_a(S, a):
    m, z = modelos[S], saes[(S, a)]
    W = m.W.detach()
    Wn = W / (W.norm(dim=0) + 1e-8)
    cos = (Wn.T @ z.direcciones()).detach()
    orden = cos.argmax(1).argsort()            # ordeno rasgos por el latente que mejor los captura
    r = evaluar(m, z, S)
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8), gridspec_kw={"width_ratios": [1.2, 1.5, 0.7]})
    ax[0].imshow(W.abs(), aspect="auto", cmap="viridis"); ax[0].set_title("Neuronas × rasgos  |W|")
    ax[0].set_xlabel("rasgo verdadero"); ax[0].set_ylabel("neurona")
    ax[1].imshow(cos[orden].clamp(min=0), aspect="auto", cmap="viridis", vmin=0, vmax=1)
    ax[1].set_title("Coseno rasgo verdadero × latente SAE"); ax[1].set_xlabel("latente"); ax[1].set_ylabel("rasgo (ordenado)")
    ax[2].bar(["neuronas", "latentes"], [r["pureza_neurona"], r["pureza_latente"]], color=["#888", "#2a9d8f"])
    ax[2].set_ylim(0, 1); ax[2].set_title("Pureza media")
    plt.tight_layout(); plt.show()
    print(f"rasgos representados: {r['representados']}/{N_RASGOS} · recupera: {r['recupera']:.2f} · "
          f"latentes vivos: {r['vivos']}/{K_DICT} · activos por muestra (L0): {r['l0']:.1f} · error relativo: {r['error_rel']:.3f}")

### 11 · b · Invocar el panel A
Crea los dos menús y conecta la función `panel_a` con `interactive_output`. Cambia un menú y el gráfico se redibuja sin reentrenar nada, porque usa los modelos ya entrenados de la sección 8.

In [ ]:
w_S = widgets.Dropdown(options=ESPARSIDADES, value=0.97, description="Esparsidad")
w_a = widgets.Dropdown(options=ALPHAS, value=0.3, description="α (L1)")
display(widgets.VBox([widgets.HBox([w_S, w_a]), widgets.interactive_output(panel_a, {"S": w_S, "a": w_a})]))

## 12 · Panel interactivo B · Dirigir una característica
Elige un rasgo, cuánto lo fuerzo (múltiplo de su activación máxima; 0 equivale a apagarlo) y *por dónde* lo fuerzo: por el latente del SAE que mejor lo representa o por la neurona que más pesa en él. Las barras muestran cuánto cambia en promedio cada una de las 12 salidas (naranja = el objetivo). Es el análogo de fijar la característica "Golden Gate Bridge" en Claude; aquí, además, sé la respuesta correcta porque conozco los rasgos. Un rasgo que el modelo no representa (con esparsidad 0.9, el 3) no tiene latente propio: no lo elijas.

### 12 · a · Función que dibuja el panel
Define `panel_b(...)`: llama a `dirigir` (sección 7 · b) con los parámetros elegidos y dibuja el cambio medio en las 12 salidas. Esta celda solo define la función.

In [ ]:
def panel_b(S, a, rasgo, factor, metodo):
    m, z = modelos[S], saes[(S, a)]
    delta, sel = dirigir(m, z, S, rasgo, factor, metodo)
    colores = ["#e76f51" if i == rasgo else "#8d99ae" for i in range(N_RASGOS)]
    plt.figure(figsize=(8, 3.2))
    plt.bar(range(N_RASGOS), delta, color=colores)
    plt.axhline(0, color="k", lw=0.6)
    plt.xticks(range(N_RASGOS)); plt.xlabel("rasgo de salida (naranja = objetivo)"); plt.ylabel("cambio medio")
    plt.title(f"Dirigir rasgo {rasgo} vía {metodo} · selectividad = {sel:.2f}")
    plt.tight_layout(); plt.show()

### 12 · b · Invocar el panel B
Crea el menú de esparsidad, α y rasgo, el slider del factor y el botón SAE/Neurona, y los conecta con `panel_b`. Mueve el slider de 0 a 10 y compara las dos vías con el mismo rasgo.

In [ ]:
b_S = widgets.Dropdown(options=ESPARSIDADES, value=0.97, description="Esparsidad")
b_a = widgets.Dropdown(options=ALPHAS, value=0.3, description="α (L1)")
b_r = widgets.Dropdown(options=list(range(N_RASGOS)), value=1, description="Rasgo")
b_f = widgets.FloatSlider(value=3.0, min=0.0, max=10.0, step=0.5, description="× máx.")
b_m = widgets.ToggleButtons(options=["SAE", "Neurona"], description="Vía")
display(widgets.VBox([widgets.HBox([b_S, b_a, b_r]), widgets.HBox([b_f, b_m]),
                      widgets.interactive_output(panel_b, {"S": b_S, "a": b_a, "rasgo": b_r, "factor": b_f, "metodo": b_m})]))

## 13 · Steering en todos los rasgos
### 13 · a · Tabla de selectividad
Para la semilla 0, α = 0.3 y factor 3, calcula la selectividad de cada rasgo representado con el SAE y con la neurona, en los mundos de esparsidad 0.9 y 0.97. Muestra primero el promedio y la desviación por esparsidad y luego la tabla completa.

In [ ]:
filas_st = []
for S in (0.9, 0.97):
    for i in range(N_RASGOS):
        if modelos[S].W.detach()[:, i].norm() > 0.5:
            filas_st.append({"esparsidad": S, "rasgo": i,
                             "selectividad_SAE": dirigir(modelos[S], saes[(S, 0.3)], S, i, 3.0, "SAE")[1],
                             "selectividad_neurona": dirigir(modelos[S], saes[(S, 0.3)], S, i, 3.0, "Neurona")[1]})
steer = pd.DataFrame(filas_st)
display(steer.groupby("esparsidad")[["selectividad_SAE", "selectividad_neurona"]].agg(["mean", "std"]).round(3))
display(steer.round(3))

### 13 · b · Gráfico: selectividad por rasgo
Barras de la selectividad del SAE y de la neurona para cada rasgo. Cuando la barra de la neurona casi no existe, fijar esa neurona no movió el rasgo objetivo (movió otros).

In [ ]:
fig_steer, ejes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for eje, (S, g_) in zip(ejes, steer.groupby("esparsidad")):
    x_ = np.arange(len(g_))
    eje.bar(x_ - 0.2, g_.selectividad_SAE, 0.4, label="SAE")
    eje.bar(x_ + 0.2, g_.selectividad_neurona, 0.4, label="Neurona")
    eje.set_xticks(x_); eje.set_xticklabels(g_.rasgo)
    eje.set_title(f"Selectividad, esparsidad {S} (factor 3)"); eje.set_xlabel("rasgo dirigido"); eje.legend()
ejes[0].set_ylabel("selectividad")
plt.tight_layout(); plt.show()

### 13 · c · Guardar el steering
Guarda `resultados_steering_semilla0.csv` y `fig2_steering_selectividad.png`.

In [ ]:
steer.round(4).to_csv(DATOS / "resultados_steering_semilla0.csv", index=False)
fig_steer.savefig(CAPTURAS / "fig2_steering_selectividad.png", dpi=150)
print("Guardados:", DATOS / "resultados_steering_semilla0.csv", "y", CAPTURAS / "fig2_steering_selectividad.png")

## 14 · Qué encontré (resultados de la ejecución en Colab, semilla 0 salvo la sección 10)

**H1 · Superposición: se cumple en el sentido esperado.** Con esparsidad 0.5 el modelo de 4 neuronas solo representa 8 de los 12 rasgos; con 0.9 representa 11 y con 0.97 los 12. Las neuronas quedan mezcladas: su pureza es 0.55, 0.45 y 0.42 (1 = una neurona, un rasgo). Mientras más esparso el mundo, más rasgos caben y más polisemánticas son las neuronas.

**H2 · Recuperación: se cumple con un α intermedio, y no con los extremos.** Con esparsidad 0.9 y 0.97 y α = 0.3, el SAE recupera 11/11 y 12/12 rasgos (semilla 0), con pureza de latentes 0.85 y 0.89 frente a 0.45 y 0.42 de las neuronas. Sin penalización (α = 0) recupera solo 0.27 y 0.25 de los rasgos: sin L1 el diccionario reconstruye bien pero no se separa. Con α = 1 la reconstrucción se degrada (error relativo ≈ 0.48 frente a ≈ 0.10) y en el mundo menos esparso (0.9) la recuperación baja a 0.91. Con esparsidad 0.5 el SAE no ayuda (recuperación máxima 0.375, pureza de latentes 0.56 frente a 0.55 de las neuronas): si los rasgos no son esparsos, no hay nada que separar.

**Robustez (5 semillas, esparsidad 0.97, α = 0.3).** La recuperación perfecta de la semilla 0 fue el mejor caso. El promedio es **0.80 (desviación 0.14)**, con valores entre 0.58 y 0.92. La pureza de latentes promedia 0.90 (±0.04) frente a 0.47 (±0.02) de las neuronas. Por eso reporto 0.80 y no 1.00 como cifra principal.

**H3 · Dirigir: se cumple en promedio, con matices.** Al fijar el latente al triple de su máximo, la selectividad promedio es 0.81 (±0.10) con el SAE y 0.23 (±0.03) con la neurona. Con esparsidad 0.9 la selectividad del SAE supera 1 en varios rasgos (hasta 1.42): el cambio en el rasgo objetivo es mayor que la suma de los cambios en los demás. Con la neurona, en varios rasgos la selectividad es ≈ 0 (no cambia el objetivo; cambian otros). Ni con el SAE el efecto es limpio: la selectividad queda por debajo de 1 en el mundo más esparso, es decir, siempre hay efectos colaterales.

**Lo que no puedo concluir.** Es un mundo sintético de 12 rasgos independientes y de igual importancia, con una sola capa. Mostrar que el método funciona aquí no demuestra que funcione en un modelo de lenguaje, donde nadie conoce los rasgos verdaderos.

## 15 · Auditoría, alineación y sesgos: qué cambia y qué riesgos aparecen

Esta sección usa lo que mostró el experimento y lo que dicen las fuentes (ver referencias). No afirmo nada que no esté en ellas.

**Auditoría.** Si el modelo tuviera rasgos que se pudieran leer, un auditor podría revisar *qué* se activa y no solo *qué* responde. Templeton et al. (2024) reportan en Claude 3 Sonnet rasgos asociados a código inseguro, sesgos, adulación, engaño y contenido peligroso. Pero los mismos autores advierten que son hallazgos preliminares y que existir un rasgo no demuestra que sirva para seguridad. En mi experimento la lectura mejora (pureza 0.90 frente a 0.47), pero la recuperación promedia 0.80, no 1: un auditor que solo mira lo que el SAE encontró puede estar mirando un diccionario incompleto. Templeton et al. señalan una limitación del mismo tipo (cobertura incompleta de rasgos).

**Alineación.** Arditi et al. (2024) muestran que el rechazo de peticiones dañinas está mediado por una única dirección en 13 modelos de chat de código abierto: se puede reforzar o quitar. O'Brien et al. (2024) fijan un rasgo de rechazo con un SAE y aumenta el rechazo de lo inseguro, pero con costo: más rechazo de peticiones seguras y degradación en MMLU, TruthfulQA y GSM8K. Wu et al. (2025, AxBench) encuentran que en steering el prompting y el ajuste fino superan a los SAE; Jørgensen y Hansen (2026) responden que con selección supervisada de rasgos los SAE se acercan. La evidencia es mixta. Mi selectividad por debajo de 1 es la versión pequeña del mismo fenómeno: intervenir en un rasgo mueve otros.

**Sesgos.** Detectar un rasgo de sesgo (por ejemplo el de género en profesiones que reportan Templeton et al.) permite medirlo y, en teoría, atenuarlo. Pero atenuarlo con una intervención interna tiene los mismos efectos colaterales que los de la sección 14, y un rasgo "de sesgo" encontrado no garantiza que sea el único canal por el que el modelo lo expresa.

**Riesgos éticos de poder dirigir.**
1. *El control es de doble uso.* La misma dirección que permite reforzar el rechazo permite apagarlo (Arditi et al., 2024 la usan como jailbreak de caja blanca barato). Quien tiene acceso a los pesos o a las activaciones puede quitar protecciones.
2. *Efectos colaterales no medidos.* Li y Kasneci (2026) reportan que vectores de steering, incluso dirigidos a conceptos inofensivos, pueden degradar la seguridad del modelo; O'Brien et al. (2024) observan degradación general de capacidades.
3. *Manipulación invisible.* Un modelo dirigido (como el del puente Golden Gate de Anthropic, que se identifica con el puente) cambia su comportamiento sin que el usuario lo vea. Si el desplegador fija rasgos de adulación, persuasión o ideología, el usuario no tiene cómo saberlo. Esta preocupación es mía como inferencia de esos casos; las fuentes no la formulan con esas palabras.
4. *Falsa seguridad.* Que el SAE etiquete un rasgo como "engaño" no prueba que el modelo lo use para engañar. Templeton et al. lo advierten expresamente.
5. *Quién decide qué se amplifica.* Elegir qué rasgos se refuerzan o se silencian es una decisión de valores; ninguna fuente que revisé propone un mecanismo de gobernanza para ello, y no encontré una (esto es una laguna que señalo, no un hecho).

## 16 · Limitaciones (lo que debo decir si me preguntan)
- **Mundo sintético:** 12 rasgos independientes, uniformes y con valores en [0, 1]. En un modelo real los rasgos son correlacionados, de distinta importancia y no se conocen.
- **Una sola capa y un SAE pequeño** (4 neuronas, 24 latentes). No pruebo escala: Anthropic usa millones de rasgos y reporta rasgos muertos (≈ 2 %, 35 % y 65 % en sus tres tamaños).
- **Hiperparámetros afinados a mano.** Elegí α = 0.3 y el tamaño del diccionario tras explorar; lo hice sobre el mismo tipo de datos, así que el resultado es optimista. Con α = 0 o α = 1 el resultado cae.
- **Resultados con una semilla vs. cinco:** la semilla 0 dio recuperación 1.00; el promedio de cinco semillas es 0.80.
- **Métricas propias** (recupera, pureza, selectividad). Son simples y están definidas en el cuaderno, no vienen de las fuentes; sirven para comparar dentro de este experimento.
- **Steering:** fijo la unidad a un múltiplo de su máximo, como el *clamp* de Anthropic, pero AxBench reporta que sumar un vector rinde mejor que fijar; no probé esa variante.
- **Lectura de fuentes:** cinco de las fuentes de arXiv las leí en informe estructurado vía alphaXiv y dos de 2026 solo en abstract e introducción; las páginas de Anthropic las leí en ≈ 72 % y ≈ 78 %. No cito cifras que no estén en mis notas (carpeta `fuentes/`).

## 17 · Referencias
- Elhage, N. et al. (2022). *Toy Models of Superposition*. arXiv:2209.10652.
- Cunningham, H., Ewart, A., Riggs, L., Huben, R. y Sharkey, L. (2023). *Sparse Autoencoders Find Highly Interpretable Features in Language Models*. arXiv:2309.08600.
- Bricken, T. et al. (2023). *Towards Monosemanticity: Decomposing Language Models With Dictionary Learning*. Transformer Circuits Thread.
- Templeton, A. et al. (2024). *Scaling Monosemanticity: Extracting Interpretable Features from Claude 3 Sonnet*. Transformer Circuits Thread.
- Arditi et al. (2024). *Refusal in Language Models Is Mediated by a Single Direction*. arXiv:2406.11717.
- Wu et al. (2025). *AxBench: Steering LLMs? Even Simple Baselines Outperform Sparse Autoencoders*. arXiv:2501.17148.
- O'Brien et al. (2024). *Steering Language Model Refusal with Sparse Autoencoders*. arXiv:2411.11296.
- Li y Kasneci (2026). *Safety Cost of Steering Vectors Is Separable and Reducible*. COLM 2026, arXiv:2608.08383.
- Jørgensen y Hansen (2026). *Steering LLMs? Actually, Sparse Autoencoders can outperform simple baselines*. arXiv:2605.31183.
- Clase 10 del curso INF659 (láminas y transcripción), PUCP, 2026-2.